Recognizing damage from a Hurricane based on satelite images
============================================================

This was from a binary classification problem given to me at the online [**TensorFlow certification exam**](https://www.credential.net/ab8ecc47-fd46-4331-80b7-beac7aff4284#acc.apuC3QgG)!

This is a deep-learning ML problem using TensorFlow's Keras library. Donwloaded and unzipped satelite images must be classified as True/False, based on the damage recognized in them using a ***CNN (Convolutional Neural Network)***.

In [ ]:
import urllib
import zipfile
import tensorflow as tf

In [ ]:
def download_and_extract_data():
    url = 'https://storage.googleapis.com/download.tensorflow.org/data/certificate/satellitehurricaneimages.zip'
    urllib.request.urlretrieve(url, 'satellitehurricaneimages.zip')
    with zipfile.ZipFile('satellitehurricaneimages.zip', 'r') as zip_ref:
        zip_ref.extractall()

In [ ]:
def preprocess(image, label):
    # NORMALIZE YOUR IMAGES HERE (HINT: Rescale by 1/.255)
    image = tf.keras.layers.Rescaling(1./255)(image)
    #image = tf.image.resize(image, (128, 128)) / .255
    return image, label

In [ ]:
# shows '/device:GPU:0' if GPU
tf.test.gpu_device_name()

download_and_extract_data()

IMG_SIZE = 128
BATCH_SIZE = 64

# The following code reads the training and validation data from their
# respective directories, resizes them into the specified image size
# and splits them into batches. You must fill in the image_size
# argument for both training and validation data.
# HINT: Image size is a tuple
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    directory='train/',
    image_size=(IMG_SIZE, IMG_SIZE)  # YOUR CODE HERE
    , batch_size=BATCH_SIZE)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    directory='validation/',
    image_size=(IMG_SIZE, IMG_SIZE)  # YOUR CODE HERE
    , batch_size=BATCH_SIZE)

# Normalizes train and validation datasets using the preprocess() function.
# Also makes other calls, as evident from the code, to prepare them for training.
# Do not batch or resize the images in the dataset here since it's already been done previously.
train_ds = train_ds.map(
    preprocess, num_parallel_calls=tf.data.experimental.AUTOTUNE).prefetch(
    tf.data.experimental.AUTOTUNE)
val_ds = val_ds.map(
    preprocess, num_parallel_calls=tf.data.experimental.AUTOTUNE)


In [ ]:
# Code to define the model
model = tf.keras.models.Sequential([
    # ADD LAYERS OF THE MODEL HERE
    tf.keras.layers.Conv2D(16, (3, 3), activation='relu', input_shape=(128, 128, 3)),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(512, activation='relu'),

    # If you don't adhere to the instructions in the following comments,
    # tests will fail to grade your model:
    # The input layer of your model must have an input shape of (128,128,3).
    # Make sure your last layer has 1 neuron activated by sigmoid.
    tf.keras.layers.Dense(1, activation=tf.nn.sigmoid)
])

# Code to compile and train the model
model.compile(
    # YOUR CODE HERE
    loss='binary_crossentropy',
    optimizer='rmsprop', # tf.keras.optimizers.RMSprop(learning_rate=0.001),
    metrics=['accuracy'])

history = model.fit(
    # YOUR CODE HERE
    train_ds,
    validation_data=val_ds,
    #steps_per_epoch=2,
    batch_size=BATCH_SIZE,
    epochs=15)

In [ ]:
import matplotlib.pyplot as plt

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
epochs = range(len(acc))

plt.plot(epochs, acc, 'r', label='Training accuracy')
plt.plot(epochs, val_acc, 'b', label='Validation accuracy')
plt.title('Training and validation accuracy')
plt.legend(loc=0)
plt.show()

In [ ]:
acc = history.history['loss']
val_acc = history.history['val_loss']
# epochs = range(len(acc))

plt.plot(epochs, acc, 'r', label='Training loss')
plt.plot(epochs, val_acc, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.legend(loc=0)
plt.show()

In [ ]:
model.save("mymodel.keras")